# Tutorial 5: Train NicheTrans on STARmap PLUS data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
# n_spot_types equals the number of distinct ct_top cell-type labels shared
# across training slides — one learnable center spatial token per cell type.
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension,
                   noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,
                   n_spot_types=dataset.n_spot_types)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

  Using provided cell-type annotation "ct_top" with 13 global cell types
------Calculating spatial graph...
The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------
  Global cell-type source: annotation
  Global cell types (13): ['Astro', 'CA1', 'CA2', 'CA3', 'CTX-Ex', 'DG', 'Endo', 'Inh', 'LHb', 'Micro', 'OPC', 'Oligo', 'SMC']


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 81/81	 Loss 0.581325 (0.677459)
==> Epoch 2/20
Batch 81/81	 Loss 0.456394 (0.514708)
==> Epoch 3/20
Batch 81/81	 Loss 0.351797 (0.404829)
==> Epoch 4/20
Batch 81/81	 Loss 0.303160 (0.330146)
==> Epoch 5/20
Batch 81/81	 Loss 0.262725 (0.277967)
==> Epoch 6/20
Batch 81/81	 Loss 0.236099 (0.238459)
==> Epoch 7/20
Batch 81/81	 Loss 0.190979 (0.207555)
==> Epoch 8/20
Batch 81/81	 Loss 0.193984 (0.186611)
==> Epoch 9/20
Batch 81/81	 Loss 0.178333 (0.170859)
==> Epoch 10/20
Batch 81/81	 Loss 0.174980 (0.159515)
==> Epoch 11/20
Batch 81/81	 Loss 0.130397 (0.147646)
==> Epoch 12/20
Batch 81/81	 Loss 0.161220 (0.141753)
==> Epoch 13/20
Batch 81/81	 Loss 0.097569 (0.132092)
==> Epoch 14/20
Batch 81/81	 Loss 0.123404 (0.128074)
==> Epoch 15/20
Batch 81/81	 Loss 0.091334 (0.124449)
==> Epoch 16/20
Batch 81/81	 Loss 0.110734 (0.119485)
==> Epoch 17/20
Batch 81/81	 Loss 0.128278 (0.115671)
==> Epoch 18/20
Batch 81/81	 Loss 0.099112 (0.111818)
==> Epoch 19/20
Batch 81/81	 Loss 0.1